In [ ]:
pip install torch tqdm scikit-learn numpy matplotlib

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import json
import re
import random
import math
import time
from tqdm.notebook import tqdm
from collections import defaultdict, Counter
import os

# --- Configuration & Hyperparameters ---
JSON_PATH = 'atis.json'
MODEL_SAVE_PATH = 'seq2seq_model_no_torchtext.pt'
PREDICTIONS_SAVE_PATH_Q = 'test_predictions_question_split.json'
PREDICTIONS_SAVE_PATH_QUERY = 'test_predictions_query_split.json'

# Model Hyperparameters
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
N_LAYERS = 1
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3

# Training Hyperparameters
N_EPOCHS = 1
CLIP = 1.0
BATCH_SIZE = 64
LEARNING_RATE = 0.001
TEACHER_FORCING_RATIO = 0.5

# Special Tokens
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

# Set Seed for reproducibility
SEED = 1234
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# --- 1. Custom Vocabulary Class ---

class Vocabulary:
    def __init__(self, special_tokens, min_freq=1, unk_token="<unk>"):
        self.itos = list(special_tokens)
        self.stoi = {token: i for i, token in enumerate(self.itos)}
        self._token_counts = Counter()
        self.min_freq = min_freq
        self.unk_token = unk_token
        self.unk_index = self.stoi.get(unk_token)
        if self.unk_index is None:
            raise ValueError(f"unk_token '{unk_token}' must be in special_tokens")

    def __len__(self):
        return len(self.itos)

    def add_token(self, token):
        self._token_counts[token] += 1

    def add_sentence(self, sentence_tokens):
        for token in sentence_tokens:
            self.add_token(token)

    def build(self):
        idx = len(self.itos)
        sorted_tokens = sorted(self._token_counts.items(), key=lambda item: (-item[1], item[0]))
        for token, count in sorted_tokens:
            if count >= self.min_freq and token not in self.stoi:
                self.stoi[token] = idx
                self.itos.append(token)
                idx += 1
        self.unk_index = self.stoi[self.unk_token]
        print(f"Built vocabulary with {len(self.itos)} tokens.")

    def numericalize(self, tokens):
        return [self.stoi.get(token, self.unk_index) for token in tokens]

    def lookup_token(self, index):
        if index < len(self.itos):
            return self.itos[index]
        return self.unk_token

    def lookup_tokens(self, indices):
        return [self.lookup_token(index) for index in indices]

In [3]:
# --- 2. Data Preprocessing ---

def normalize_sql(sql_query):
    return re.sub(r'\s+', ' ', sql_query).strip()

def substitute_variables(text, variables):
    substituted_text = text
    sorted_vars = sorted(variables.keys(), key=len, reverse=True)
    is_sql = any(kw in text.upper() for kw in ["SELECT", "FROM", "WHERE", "=", ">", "<"])
    for var_name in sorted_vars:
        value = str(variables[var_name])
        if is_sql:
            placeholder = f'"{var_name}"'
            try:
                float(value)
                replacement = f'"{value}"'
            except ValueError:
                replacement = f'"{value}"'
            substituted_text = substituted_text.replace(placeholder, replacement)
        else:
            placeholder = var_name
            replacement = value
            substituted_text = substituted_text.replace(placeholder, replacement)
    return substituted_text

def load_and_process_data(json_path):
    print(f"Loading and processing data from {json_path}...")
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Error: {json_path} not found.")
    with open(json_path, 'r') as f:
        data = json.load(f)
    processed_examples = []
    skipped_count = 0
    for item in data:
        sql_templates = item['sql']
        if not sql_templates:
            skipped_count += 1
            continue
        shortest_sql_template = min(sql_templates, key=lambda x: (len(x), x))
        all_sql_templates = item['sql']
        query_split = item['query-split']
        for sentence_info in item['sentences']:
            nl_question_template = sentence_info['text']
            sentence_vars = sentence_info['variables']
            question_split = sentence_info['question-split']
            try:
                final_question = substitute_variables(nl_question_template, sentence_vars)
                final_sql_shortest = substitute_variables(shortest_sql_template, sentence_vars)
                final_sql_shortest_normalized = normalize_sql(final_sql_shortest)
                all_final_sql_options = set()
                for sql_template in all_sql_templates:
                    final_sql_option = substitute_variables(sql_template, sentence_vars)
                    all_final_sql_options.add(normalize_sql(final_sql_option))
                all_final_sql_options_list = sorted(list(all_final_sql_options))
                processed_examples.append({
                    "question": final_question,
                    "sql_shortest": final_sql_shortest_normalized,
                    "sql_options": all_final_sql_options_list,
                    "question_split": question_split,
                    "query_split": query_split,
                })
            except Exception as e:
                print(f"Warning: Skipping example due to error: {e}")
                skipped_count += 1
    print(f"Finished processing. Loaded {len(processed_examples)} examples. Skipped {skipped_count}.")
    return processed_examples

def split_data(processed_data, split_type='question'):
    key = f"{split_type}_split"
    splits = defaultdict(list)
    for example in processed_data:
        splits[example[key]].append(example)
    print(f"Splitting data by '{key}': Train={len(splits['train'])}, Dev={len(splits['dev'])}, Test={len(splits['test'])}")
    return splits['train'], splits['dev'], splits['test']

In [4]:
# --- 3. Tokenization ---

def tokenize_en(text):
    return text.lower().split(' ')

def tokenize_sql(text):
    text = text.replace('(', ' ( ').replace(')', ' ) ').replace(',', ' , ').replace('=', ' = ').replace('!=', ' != ').replace('"', ' " ')
    return text.lower().split(' ')

In [5]:
# --- 4. Build Vocabularies ---

def build_vocabularies_manual(train_data, src_tokenizer, trg_tokenizer):
    print("Building vocabularies manually...")
    source_vocab = Vocabulary(SPECIAL_TOKENS, min_freq=2, unk_token=UNK_TOKEN)
    target_vocab = Vocabulary(SPECIAL_TOKENS, min_freq=1, unk_token=UNK_TOKEN)

    print("Adding source tokens...")
    for example in tqdm(train_data, desc="Source Vocab"):
        source_vocab.add_sentence(src_tokenizer(example['question']))

    print("Adding target tokens...")
    for example in tqdm(train_data, desc="Target Vocab"):
        target_vocab.add_sentence(trg_tokenizer(example['sql_shortest']))

    print("Building source vocab index...")
    source_vocab.build()
    print("Building target vocab index...")
    target_vocab.build()

    return source_vocab, target_vocab

In [6]:
# --- 5. Dataset & DataLoader ---

class AtisDatasetManual(Dataset):
    def __init__(self, data, src_vocab, trg_vocab, src_tokenizer, trg_tokenizer):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer
        self.sos_idx = trg_vocab.stoi[SOS_TOKEN]
        self.eos_idx = trg_vocab.stoi[EOS_TOKEN]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src_tokens = self.src_tokenizer(item['question'])
        trg_tokens = self.trg_tokenizer(item['sql_shortest'])

        src_indices = self.src_vocab.numericalize(src_tokens)
        trg_indices = [self.sos_idx] + self.trg_vocab.numericalize(trg_tokens) + [self.eos_idx]

        return torch.tensor(src_indices, dtype=torch.long), torch.tensor(trg_indices, dtype=torch.long)

def collate_fn_manual(batch, pad_idx_src, pad_idx_trg):
    src_batch, trg_batch = [], []
    for src_item, trg_item in batch:
        src_batch.append(src_item)
        trg_batch.append(trg_item)

    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=pad_idx_src)
    trg_padded = pad_sequence(trg_batch, batch_first=True, padding_value=pad_idx_trg)

    return {'src': src_padded, 'trg': trg_padded}

In [7]:
# --- 6. Model Architecture ---

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout if n_layers > 1 else 0, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
        return outputs

In [8]:
# --- 7. Training & Evaluation Loops ---

def train_epoch(model, iterator, optimizer, criterion, clip, teacher_forcing_ratio):
    model.train()
    epoch_loss = 0
    pbar = tqdm(iterator, desc=f"Training Epoch {epoch+1}/{N_EPOCHS}", leave=False)
    for batch in pbar:
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        output_dim = output.shape[-1]
        output_flat = output[:, 1:].reshape(-1, output_dim)
        trg_flat = trg[:, 1:].reshape(-1)
        loss = criterion(output_flat, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    return epoch_loss / len(iterator)

def evaluate_loss(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        pbar = tqdm(iterator, desc="Evaluating Loss", leave=False)
        for batch in pbar:
            src = batch['src'].to(device)
            trg = batch['trg'].to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output_flat = output[:, 1:].reshape(-1, output_dim)
            trg_flat = trg[:, 1:].reshape(-1)
            loss = criterion(output_flat, trg_flat)
            epoch_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
    return epoch_loss / len(iterator)

In [9]:
# --- 8. Prediction & Accuracy Calculation ---

def predict_sequence_manual(model, src_tensor, src_vocab, trg_vocab, device, max_len=100):
    model.eval()
    sos_idx = trg_vocab.stoi[SOS_TOKEN]
    eos_idx = trg_vocab.stoi[EOS_TOKEN]

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
        trg_indexes = [sos_idx]
        input_token = torch.tensor([sos_idx], dtype=torch.long, device=device)

        for _ in range(max_len):
            output, hidden, cell = model.decoder(input_token, hidden, cell)
            pred_token_idx = output.argmax(1).item()
            trg_indexes.append(pred_token_idx)
            if pred_token_idx == eos_idx:
                break
            input_token = torch.tensor([pred_token_idx], dtype=torch.long, device=device)

    trg_tokens = trg_vocab.lookup_tokens(trg_indexes[1:])
    return trg_tokens

def generate_predictions_and_calculate_accuracy(model, test_data, src_vocab, trg_vocab, src_tokenizer, device, predictions_save_path, max_len=100):
    print(f"Generating predictions and calculating accuracy for {predictions_save_path}...")
    correct_count = 0
    total_count = len(test_data)
    predictions_output = []

    pad_token_str = PAD_TOKEN
    eos_token_str = EOS_TOKEN
    unk_token_str = UNK_TOKEN

    for i, example in enumerate(tqdm(test_data, desc="Testing Accuracy")):
        src_text = example['question']
        true_sql_options = example['sql_options']

        src_tokens = src_tokenizer(src_text)
        src_indices = src_vocab.numericalize(src_tokens)
        src_tensor = torch.tensor(src_indices, dtype=torch.long).unsqueeze(0).to(device)

        predicted_tokens = predict_sequence_manual(model, src_tensor, src_vocab, trg_vocab, device, max_len)

        predicted_sql_parts = []
        for token in predicted_tokens:
            if token == eos_token_str:
                break
            if token != unk_token_str and token != pad_token_str:
                predicted_sql_parts.append(token)

        predicted_sql = " ".join(predicted_sql_parts)
        predicted_sql = predicted_sql.replace(' ( ', '(').replace(' ) ', ')').replace(' , ', ',').replace(' = ', '=').replace(' != ', '!=').replace(' " ', '"')
        predicted_sql_normalized = normalize_sql(predicted_sql)

        predictions_output.append({
            "question": src_text,
            "predicted_sql": predicted_sql_normalized,
            "true_sql_options": true_sql_options
        })

        if predicted_sql_normalized in true_sql_options:
            correct_count += 1

    try:
        with open(predictions_save_path, 'w') as f:
            json.dump(predictions_output, f, indent=4)
        print(f"Saved predictions to {predictions_save_path}")
    except Exception as e:
        print(f"Error saving predictions to {predictions_save_path}: {e}")

    accuracy = correct_count / total_count if total_count > 0 else 0
    return accuracy, predictions_save_path

In [10]:
# pip install -U ipywidgets

In [14]:
# --- 9. Main Execution Block ---

def run_training_and_evaluation():
    processed_data = load_and_process_data(JSON_PATH)
    q_train_data, q_dev_data, q_test_data = split_data(processed_data, 'question')
    query_train_data, query_dev_data, query_test_data = split_data(processed_data, 'query')

    source_vocab, target_vocab = build_vocabularies_manual(q_train_data, tokenize_en, tokenize_sql)
    SRC_PAD_IDX = source_vocab.stoi[PAD_TOKEN]
    TRG_PAD_IDX = target_vocab.stoi[PAD_TOKEN]
    INPUT_DIM = len(source_vocab)
    OUTPUT_DIM = len(target_vocab)

    q_train_dataset = AtisDatasetManual(q_train_data, source_vocab, target_vocab, tokenize_en, tokenize_sql)
    q_dev_dataset = AtisDatasetManual(q_dev_data, source_vocab, target_vocab, tokenize_en, tokenize_sql)
    q_test_dataset = AtisDatasetManual(q_test_data, source_vocab, target_vocab, tokenize_en, tokenize_sql)
    query_test_dataset = AtisDatasetManual(query_test_data, source_vocab, target_vocab, tokenize_en, tokenize_sql)

    collate_train = lambda batch: collate_fn_manual(batch, SRC_PAD_IDX, TRG_PAD_IDX)
    collate_eval = lambda batch: collate_fn_manual(batch, SRC_PAD_IDX, TRG_PAD_IDX)

    q_train_iterator = DataLoader(q_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_train)
    q_dev_iterator = DataLoader(q_dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_eval)

    encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
    decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
    model = Seq2Seq(encoder, decoder, device).to(device)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

    global epoch
    best_valid_loss = float('inf')
    print("\n--- Starting Training (Manual Vocab) ---")
    for epoch in range(N_EPOCHS):
        start_time = time.time()
        current_tf_ratio = TEACHER_FORCING_RATIO
        train_loss = train_epoch(model, q_train_iterator, optimizer, criterion, CLIP, current_tf_ratio)
        valid_loss = evaluate_loss(model, q_dev_iterator, criterion)
        end_time = time.time()
        epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            # Saving everything as a dictionary
            torch.save({
                'model_state_dict': model.state_dict(),
                'source_vocab': source_vocab,
                'target_vocab': target_vocab,
                'input_dim': INPUT_DIM,
                'output_dim': OUTPUT_DIM,
                'enc_emb_dim': ENC_EMB_DIM,
                'dec_emb_dim': DEC_EMB_DIM,
                'hid_dim': HID_DIM,
                'n_layers': N_LAYERS,
                'enc_dropout': ENC_DROPOUT,
                'dec_dropout': DEC_DROPOUT
            }, MODEL_SAVE_PATH) 
            
            print(f"| Epoch: {epoch+1:02} | Time: {epoch_mins:.0f}m {epoch_secs:.0f}s | Saved Best Model |")
            print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
            print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')
        else:
            print(f"| Epoch: {epoch+1:02} | Time: {epoch_mins:.0f}m {epoch_secs:.0f}s |")
            print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
            print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

    print("\n--- Training Finished ---")

    print(f"Loading best model from {MODEL_SAVE_PATH}")
    
    # --- FIX STARTS HERE ---
    # 1. Load the checkpoint dictionary
    checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=False)
    
    # 2. Extract ONLY the state_dict for the model
    model.load_state_dict(checkpoint['model_state_dict'])
    # --- FIX ENDS HERE ---

    q_test_accuracy, q_pred_file = generate_predictions_and_calculate_accuracy(
        model, q_test_data, source_vocab, target_vocab, tokenize_en, device, PREDICTIONS_SAVE_PATH_Q
    )
    print(f"\n--- Question Split Test Set ---")
    print(f"\tAccuracy: {q_test_accuracy*100:.2f}% (Predictions saved to {q_pred_file})")

    query_test_accuracy, query_pred_file = generate_predictions_and_calculate_accuracy(
        model, query_test_data, source_vocab, target_vocab, tokenize_en, device, PREDICTIONS_SAVE_PATH_QUERY
    )
    print(f"\n--- Query Split Test Set ---")
    print(f"\tAccuracy: {query_test_accuracy*100:.2f}% (Predictions saved to {query_pred_file})")

    print("\n--- Script Finished ---")
    return q_pred_file, query_split_pred_file

q_split_pred_file, query_split_pred_file = run_training_and_evaluation()

Loading and processing data from atis.json...
Finished processing. Loaded 5280 examples. Skipped 0.
Splitting data by 'question_split': Train=4347, Dev=486, Test=447
Splitting data by 'query_split': Train=4812, Dev=121, Test=347
Building vocabularies manually...
Adding source tokens...


Source Vocab:   0%|          | 0/4347 [00:00<?, ?it/s]

Adding target tokens...


Target Vocab:   0%|          | 0/4347 [00:00<?, ?it/s]

Building source vocab index...
Built vocabulary with 564 tokens.
Building target vocab index...
Built vocabulary with 573 tokens.

--- Starting Training (Manual Vocab) ---


...

| Epoch: 01 | Time: 3m 4s | Saved Best Model |
	Train Loss: 3.201 | Train PPL:  24.565
	 Val. Loss: 2.757 |  Val. PPL:  15.747

--- Training Finished ---
Loading best model from seq2seq_model_no_torchtext.pt
Generating predictions and calculating accuracy for test_predictions_question_split.json...


Testing Accuracy:   0%|          | 0/447 [00:00<?, ?it/s]

Saved predictions to test_predictions_question_split.json

--- Question Split Test Set ---
	Accuracy: 0.00% (Predictions saved to test_predictions_question_split.json)
Generating predictions and calculating accuracy for test_predictions_query_split.json...


Testing Accuracy:   0%|          | 0/347 [00:00<?, ?it/s]

Saved predictions to test_predictions_query_split.json

--- Query Split Test Set ---
	Accuracy: 0.00% (Predictions saved to test_predictions_query_split.json)

--- Script Finished ---


In [16]:
# --- 10. Case-Insensitive Accuracy Calculation ---

import json
import re
from tqdm.notebook import tqdm

def normalize_sql_lower(sql_query):
    if not isinstance(sql_query, str):
        return ""
    normalized = re.sub(r'\s+', ' ', sql_query).strip()
    return normalized.lower()

def calculate_accuracy_from_json_case_insensitive(json_filepath):
    correct_count = 0
    total_count = 0

    if json_filepath is None or not os.path.exists(json_filepath):
        print(f"Error: Predictions file '{json_filepath}' not found or not specified. Skipping accuracy calculation.")
        return None

    try:
        with open(json_filepath, 'r') as f:
            predictions_data = json.load(f)
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{json_filepath}'. Check file format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while reading the file '{json_filepath}': {e}")
        return None

    if not isinstance(predictions_data, list):
        print(f"Error: Expected JSON file '{json_filepath}' to contain a list of predictions.")
        return None

    print(f"\nCalculating Case-Insensitive Accuracy for: {json_filepath}")
    print(f"Processing {len(predictions_data)} predictions...")

    for item in tqdm(predictions_data, desc="Checking Accuracy"):
        total_count += 1

        predicted_sql = item.get("predicted_sql")
        true_sql_options = item.get("true_sql_options")

        if predicted_sql is None or not isinstance(predicted_sql, str):
            print(f"Warning: Skipping item {total_count} due to invalid 'predicted_sql'.")
            continue

        if true_sql_options is None or not isinstance(true_sql_options, list):
            print(f"Warning: Skipping item {total_count} due to invalid 'true_sql_options'.")
            continue

        normalized_predicted_lower = normalize_sql_lower(predicted_sql)

        is_match = False
        for option in true_sql_options:
            normalized_option_lower = normalize_sql_lower(option)
            if normalized_predicted_lower == normalized_option_lower:
                is_match = True
                break

        if is_match:
            correct_count += 1

    if total_count == 0:
        print("No valid prediction items found in the file.")
        return 0.0

    accuracy = (correct_count / total_count) * 100
    print(f"\nCalculation Complete for {json_filepath}:")
    print(f"  Total Predictions Checked: {total_count}")
    print(f"  Correct Predictions:     {correct_count}")
    print(f"  Accuracy (Case-Insensitive): {accuracy:.2f}%")

    return accuracy

if 'q_split_pred_file' in locals() and 'query_split_pred_file' in locals():
    q_accuracy_ci = calculate_accuracy_from_json_case_insensitive(q_split_pred_file)
    query_accuracy_ci = calculate_accuracy_from_json_case_insensitive(query_split_pred_file)
else:
    print("\nWarning: Prediction file paths not found. Did the training/evaluation cell run successfully?")
    print("Attempting to calculate using default filenames...")
    q_accuracy_ci = calculate_accuracy_from_json_case_insensitive(PREDICTIONS_SAVE_PATH_Q)
    query_accuracy_ci = calculate_accuracy_from_json_case_insensitive(PREDICTIONS_SAVE_PATH_QUERY)


Calculating Case-Insensitive Accuracy for: test_predictions_question_split.json
Processing 447 predictions...


Checking Accuracy:   0%|          | 0/447 [00:00<?, ?it/s]


Calculation Complete for test_predictions_question_split.json:
  Total Predictions Checked: 447
  Correct Predictions:     0
  Accuracy (Case-Insensitive): 0.00%
Error: Predictions file 'None' not found or not specified. Skipping accuracy calculation.
